### This is where we chain every step

In [ ]:
import sys
from pathlib import Path
import yaml
import pandas as pd
project_dir = Path.cwd().parent
sys.path.append(str(project_dir))
config_path = project_dir / "config.yaml"

from src.corrupt import random_missing_values
from src.clean import mean_fill
from src.evaluate import evaluate_continuous_recovery
from src.load import load_model_data

with config_path.open() as config_file:
    config = yaml.safe_load(config_file)

corruption_rate = config["corruption_rate"]
target_drug = config["target_drug"]
columns = config["columns"]
seed = config["seed"]

processed_dir = project_dir / "data/processed/"
drug_consumption = project_dir / f"data/processed/Drug_Consumption_{target_drug}.csv"

# load the dfs
original_df = load_model_data(drug_consumption)

# corrupt the df
corrupted_df, mask_df = random_missing_values(original_df, corruption_rate, seed)

# clean the df 
cleaned_df = mean_fill(corrupted_df, columns, {})

# evaluate the cleaned df
evaluated_metrics = evaluate_continuous_recovery(original_df, cleaned_df, mask_df, columns)

print("Continuous Recovery Metrics")
print("---------------------------")
print(f"RMSE:             {evaluated_metrics['rmse']:.4f}")
print(f"MAE:              {evaluated_metrics['mae']:.4f}")
print(f"Bias:             {evaluated_metrics['bias']:.4f}")
print(f"Variance Ratio:   {evaluated_metrics['variance_ratio']:.4f}")
print(f"Wasserstein:      {evaluated_metrics['wasserstein']:.4f}")

Continuous Recovery Metrics
---------------------------
RMSE:             1.0051
MAE:              0.8077
Bias:             -0.0161
Variance Ratio:   0.0001
Wasserstein:      0.7998
